# لیسن 01 - اے آئی ایجنٹس کا تعارف

**AI Agents for Beginners** کورس کے پہلے سبق میں خوش آمدید!

ایک **AI ایجنٹ** ایک پروگرام ہے جو بڑی زبان کا ماڈل (LLM) بطور وجدان انجن استعمال کرتا ہے اور حقیقی دنیا میں *عمل* کر سکتا ہے — APIs کو کال کرنا، ڈیٹا بیس سے سوال کرنا، یا کوڈ چلانا — تاکہ صارف کی جانب سے کسی مقصد کو حاصل کرے۔

اس نوٹ بک میں آپ اپنا پہلا ایجنٹ بنائیں گے: ایک **Travel Agent** جو تعطیلات کے مقامات کی سفارش کرتا ہے۔ اس دوران آپ سیکھیں گے کہ:

1. **Microsoft Agent Framework** استعمال کرتے ہوئے Azure AI Foundry Agent Service سے جڑیں۔
2. ایجنٹ کو ایک **ٹول** دیں — ایک سادہ Python فنکشن جسے وہ کال کر سکتا ہے۔
3. ایجنٹ کو چلائیں اور اس کے جواب کا معائنہ کریں۔
4. ایجنٹ کے جواب کو ٹوکن بہ ٹوکن اسٹریم کریں۔


## سیٹ اپ

اس نوٹ بک کو چلانے سے پہلے، یقینی بنائیں کہ آپ کے پاس:

1. **ایک Azure AI Foundry پروجیکٹ** جس میں چیٹ ماڈل تعینات ہے (مثلاً `gpt-4o-mini`)۔
2. **Azure CLI کے ساتھ لاگ ان کیا ہوا ہو** — اپنے ٹرمینل میں `az login` چلائیں۔
3. **ضروری ماحولیاتی متغیرات مرتب کیے ہوں:**
   - `AZURE_AI_PROJECT_ENDPOINT` — آپ کے Azure AI Foundry پروجیکٹ کا اینڈ پوائنٹ۔
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — آپ کے تعینات کردہ ماڈل کا نام۔

نیچے دیا گیا خلیہ وہ Python پیکیجز انسٹال کرتا ہے جن کی آپ کو ضرورت ہے۔


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## اپنا پہلا ایجنٹ بنانا

ایک ایجنٹ کو دو چیزیں چاہیے:

- **ہدایتیں** جو اسے بتائیں *وہ کون ہے* اور *کیسی کارکردگی دکھانی ہے* (ایک نظام کا اشارہ)۔
- **اوزار** — Python کے فنکشنز جنہیں `@tool` سے سجا دیا گیا ہو، جنہیں ایجنٹ معلومات حاصل کرنے یا کارروائیاں انجام دینے کے لیے بلا سکتا ہے۔

نیچے ہم ایک سادہ اوزار کی تعریف کرتے ہیں جو مقبول تعطیلاتی مقامات کی فہرست واپس دیتا ہے۔ جب صارف سفر کی سفارشات مانگے گا تو ایجنٹ اس اوزار کو استعمال کرے گا۔


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## جوابات کا سلسلہ وار آنا

ایک زیادہ تعاملی تجربے کے لیے آپ ایجنٹ کے جواب کو **اسٹریم** کر سکتے ہیں۔ مکمل جواب کے انتظار کرنے کے بجائے، ایجنٹ جوں جوں متن کے حصے بناتا ہے، وہ فراہم کرتا ہے۔ یہ خاص طور پر چیٹ انٹرفیسز میں مفید ہے جہاں آپ کو آؤٹ پٹ کو حقیقی وقت میں دکھانا ہوتا ہے۔


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## خلاصہ

اس سبق میں آپ نے سیکھا کہ:

- **ایک فراہم کنندہ بنائیں** جو `AzureAIProjectAgentProvider` کے ذریعے Azure AI Foundry Agent Service سے جڑتا ہے۔
- **ایک ٹول کی تعریف کریں** `@tool` ڈیکوریٹر استعمال کرتے ہوئے تاکہ ایجنٹ آپ کے Python فنکشنز کو کال کر سکے۔
- **ایجنٹ چلائیں** صارف کے پیغام کے ساتھ اور اس کا جواب پرنٹ کریں۔
- **جوابات کو اسٹریم کریں** تاکہ ریئل ٹائم آؤٹ پٹ حاصل ہو۔

اگلے سبق میں ہم ایجنٹک فریم ورکس کو مزید گہرائی سے دیکھیں گے اور سیکھیں گے کہ ایجنٹس کو زیادہ طاقتور ٹولز اور کثیر مرحلہ وار استدلال کی صلاحیتیں کیسے دی جائیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
